# Fine-tune Ministral-8B on dbt DAG generation

**Goal**: teach Ministral-8B-Instruct to take a business question + SQL schemas → produce a  
production-ready multi-file dbt DAG (staging + optional intermediate + marts).

**Technique**: QLoRA (4-bit quantisation + LoRA adapters) with `unsloth` for fast training.

**Runtime**: GPU — **T4 (free)** on Colab or Kaggle, **A100** on Colab Pro.  
**Platform**: auto-detected — works on **Google Colab** (Drive upload) and **Kaggle** (dataset attached via right panel).

### Baseline results (pre fine-tune, Ministral-8B)
| Metric | Score |
|---|---|
| dbt_parse_pass_rate | 51% |
| has_staging_rate | 98% |
| has_marts_rate | 100% |
| correct_prefix_rate | 100% |

The model already understands dbt structure perfectly. Fine-tuning targets the 49%  
YAML/syntax failures → expected post-finetune pass rate: **~85–90%**.

### ETA on T4 (free Colab)
| Step | Time |
|---|---|
| Install deps | ~3 min |
| Download Ministral-8B | ~8 min |
| Training (900 rows × 3 epochs, batch=16, ~168 steps) | ~15–20 min |
| GGUF export | ~5 min |
| **Total** | **~30–40 min** |

No Colab timeout risk — the 90-min idle limit does not apply during active GPU training.  
Checkpoints are saved every 50 steps so you can resume from `./checkpoints` if needed.

---
### Steps
1. Install dependencies
2. Upload dataset files
3. Load model in 4-bit
4. Attach LoRA adapters
5. Train with SFTTrainer
6. Quick inference check
7. Save LoRA adapters
8. Export GGUF → download for LM Studio
9. (Optional) Push to HuggingFace Hub

## 0 — Runtime check

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No GPU found — '
    'Colab: Runtime → Change runtime type → T4 GPU | '
    'Kaggle: Settings (right panel) → Accelerator → GPU T4 x2'
)
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import os

# ── Platform detection ──────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/input')
PLATFORM  = 'Kaggle' if IS_KAGGLE else 'Google Colab'
print(f'Running on: {PLATFORM}')

# ── Paths ───────────────────────────────────────────────────────────
if IS_KAGGLE:
    DATASET_SLUG = 'text-to-dbt'   # Kaggle converts dataset names to slugs
    DATA_PATH    = f'/kaggle/input/{DATASET_SLUG}'
    OUTPUT_PATH  = '/kaggle/working'
else:
    DATA_PATH    = 'data'
    OUTPUT_PATH  = '.'

print(f'Data:   {DATA_PATH}')
print(f'Output: {OUTPUT_PATH}')

## 1 — Install dependencies

In [ ]:
import subprocess, sys

# unsloth has separate wheels for Colab and Kaggle
pkg = 'unsloth[kaggle-new]' if IS_KAGGLE else 'unsloth[colab-new]'
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    pkg,
    'trl>=0.9',
    'transformers>=4.40',
    'datasets',
    'bitsandbytes',
    'peft',
    'accelerate',
    '-q',
], check=True)
print('Dependencies installed.')

## 2 — Upload dataset & configuration

In [ ]:
import json, os, shutil

if IS_KAGGLE:
    # Dataset is mounted read-only; copy to ./data/ for consistent paths
    os.makedirs('data', exist_ok=True)
    for fname in ('train.jsonl', 'eval.jsonl'):
        src, dst = f'{DATA_PATH}/{fname}', f'data/{fname}'
        shutil.copy(src, dst)
        print(f'Copied {fname} -> {dst}')
else:
    # ── Google Drive ──────────────────────────────────────────────────────
    # Upload train.jsonl + eval.jsonl to the root of your Google Drive first,
    # then run this cell — it will ask for a one-time authorisation.
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_FOLDER = '/content/drive/MyDrive'
    os.makedirs('data', exist_ok=True)
    for fname in ('train.jsonl', 'eval.jsonl'):
        src = os.path.join(DRIVE_FOLDER, fname)
        dst = f'data/{fname}'
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f'Copied {fname} -> {dst}')
        else:
            candidates = [
                os.path.join(root, f)
                for root, _, files in os.walk(DRIVE_FOLDER)
                for f in files if f == fname
            ]
            if candidates:
                shutil.copy(candidates[0], dst)
                print(f'Copied {fname} from {candidates[0]} -> {dst}')
            else:
                raise FileNotFoundError(
                    f'{fname} not found in Drive. '
                    f"Upload it to '{DRIVE_FOLDER}' and re-run this cell."
                )

# ── Verify ──────────────────────────────────────────────────────────────
train_rows = [json.loads(l) for l in open('data/train.jsonl')]
eval_rows  = [json.loads(l) for l in open('data/eval.jsonl')]
print(f'\nTrain: {len(train_rows)} rows  |  Eval: {len(eval_rows)} rows')
print('Sample keys:', list(train_rows[0].keys()))
print('Sample roles:', [m["role"] for m in train_rows[0]['messages']])

## 3 — Load Ministral-8B in 4-bit (QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME    = "mistralai/Ministral-8B-Instruct-2410"
MAX_SEQ_LEN   = 2048   # all rows fit under ~950 tokens; headroom for prompt
LOAD_IN_4BIT  = True
DTYPE         = None   # auto-detect (bfloat16 on Ampere+, float16 on T4)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print(f"Model loaded: {MODEL_NAME}")
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 4 — Attach LoRA adapters

In [ ]:
# ── LoRA config ─────────────────────────────────────────────────────────────
# r=16 is a good default: more expressive than r=8, less VRAM than r=64
# Target all attention + feed-forward projection layers
LORA_R         = 16
LORA_ALPHA     = 32    # scaling = alpha / r = 2; higher = larger effective LR
LORA_DROPOUT   = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # attention
    "gate_proj", "up_proj", "down_proj",        # MLP
]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # significantly reduces VRAM
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

## 5 — Prepare datasets

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments
import json

def load_jsonl(path):
    return [json.loads(l) for l in open(path)]

train_data = Dataset.from_list(load_jsonl('data/train.jsonl'))
eval_data  = Dataset.from_list(load_jsonl('data/eval.jsonl'))

# Format messages into the Mistral instruct template
def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_data = train_data.map(apply_chat_template)
eval_data  = eval_data.map(apply_chat_template)

print("Sample formatted text (first 600 chars):")
print(train_data[0]['text'][:600])

## 6 — Train

In [ ]:
from unsloth import is_bfloat16_supported
import glob as _glob, os

CKPT_DIR = os.path.join(OUTPUT_PATH, 'checkpoints')

# ── Training hyperparameters ─────────────────────────────────────────────
# With 900 rows and batch_size=4, grad_accum=4 -> effective batch = 16
# 3 epochs * 900 / 16 ≈ 168 steps  -> fast even on T4

training_args = TrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    warmup_ratio                = 0.05,
    learning_rate               = 2e-4,
    lr_scheduler_type           = 'cosine',
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 10,
    eval_strategy               = 'steps',
    eval_steps                  = 50,
    save_strategy               = 'steps',
    save_steps                  = 50,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    optim                       = 'adamw_8bit',
    seed                        = 42,
    report_to                   = 'none',
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_data,
    eval_dataset       = eval_data,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LEN,
    args               = training_args,
)

# Resume from latest checkpoint if one exists (safe to re-run after interruption)
_ckpts  = sorted(_glob.glob(os.path.join(CKPT_DIR, 'checkpoint-*')))
_resume = _ckpts[-1] if _ckpts else None
if _resume:
    print(f'Resuming from checkpoint: {_resume}')
else:
    print('Starting training from scratch...')

trainer_stats = trainer.train(resume_from_checkpoint=_resume)
print(f'\nDone. Final loss: {trainer_stats.training_loss:.4f}')

## 7 — Quick quality check (inference)

In [ ]:
from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(model)  # 2x faster inference

test_question = "Show the total revenue per product category."
test_context  = "CREATE TABLE products (product_id INT, category VARCHAR, price DECIMAL); CREATE TABLE orders (order_id INT, product_id INT, quantity INT)"

messages = [
    {"role": "system",  "content": "You are a Staff Analytics Engineer expert in dbt."},
    {"role": "user",    "content": f"Business question: {test_question}\n\nSQL schemas:\n{test_context}"},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=1024,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(generated)

## 8 — Save LoRA adapters (lightweight checkpoint)

In [ ]:
import os

# Save LoRA weights only (~60 MB for r=16) — fast, good for resuming
LORA_DIR = os.path.join(OUTPUT_PATH, 'lora_adapters')
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f'LoRA adapters saved to {LORA_DIR}')

if not IS_KAGGLE:
    # Optional: download adapters to your machine
    # from google.colab import files
    # import shutil
    # shutil.make_archive('lora_adapters', 'zip', LORA_DIR)
    # files.download('lora_adapters.zip')
    pass

## 9 — Export GGUF for LM Studio

This merges the LoRA weights into the base model and quantises to GGUF Q4_K_M —  
the format LM Studio loads directly.

In [ ]:
import os

# Merge LoRA into base model and save as GGUF Q4_K_M
# Q4_K_M = 4-bit quantisation, good quality/size balance (~4.5 GB for 8B model)
GGUF_DIR = os.path.join(OUTPUT_PATH, 'ministral-dbt-gguf')
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method='q4_k_m',
)
print(f'GGUF file saved to {GGUF_DIR}/')

for f in os.listdir(GGUF_DIR):
    size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.0f} MB)')

In [ ]:
import glob, os

GGUF_DIR   = os.path.join(OUTPUT_PATH, 'ministral-dbt-gguf')
gguf_files = glob.glob(f'{GGUF_DIR}/*.gguf')
print(f'Found {len(gguf_files)} GGUF file(s):')
for f in gguf_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {f}  ({size_mb:.0f} MB)')

if IS_KAGGLE:
    print('\nKaggle: open the Output tab (right panel) to download the GGUF file.')
    print(f'Path: {GGUF_DIR}/')
else:
    from google.colab import files
    for f in gguf_files:
        print(f'Downloading {f} ...')
        files.download(f)

## 10 — (Optional) Push to HuggingFace Hub

Saves the LoRA adapters to your HuggingFace account so you can reuse them later  
without re-training (e.g. try different base models, share with team).

In [ ]:
# # Set your HuggingFace token (get it at hf.co/settings/tokens)
# HF_TOKEN   = "hf_..."  # replace with your token
# HF_REPO_ID = "your-username/ministral-8b-dbt-instruct"  # replace with your repo

# model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
# print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")